# Plankton v2 — Pipeline optimisée macro-F1

Cette version inclut : split stratifié K-Fold, perte adaptée au déséquilibre, modèle backbone plus fort, augmentations avancées, progressive resizing, AMP, TTA et ensembling des folds pour la soumission.

In [1]:
# (Optionnel) Installe les dépendances dans le kernel du notebook
%pip install -q scikit-learn pandas tqdm

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import random
import time
from dataclasses import dataclass
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset, WeightedRandomSampler
from torchvision import datasets, models, transforms
from sklearn.metrics import f1_score
from tqdm.auto import tqdm

@dataclass
class CFG:
    seed: int = 42

    train_dir: str = './train'
    test_dir: str = './test'
    ckpt_dir: str = './checkpoints_v2'

    val_ratio: float = 0.10
    epochs: int = 14
    batch_size: int = 48
    num_workers: int = 4

    lr: float = 2e-4
    weight_decay: float = 1e-4
    grad_accum_steps: int = 1
    early_stopping_patience: int = 6

    # options performance
    img_size: int = 288
    use_mixup: bool = True
    mixup_alpha: float = 0.3
    tta: bool = True

CFG = CFG()

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything(CFG.seed)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)
if device.type != 'cuda':
    raise RuntimeError('CUDA non disponible sur ce kernel.')

torch.backends.cudnn.benchmark = True
Path(CFG.ckpt_dir).mkdir(exist_ok=True, parents=True)

device: cuda


c:\Users\thiev\OneDrive\Bureau\Plankton\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Chargement datasets (pipeline stable type Plankton.ipynb)
train_root = Path(CFG.train_dir)
test_root = Path(CFG.test_dir)
assert train_root.exists(), f'train introuvable: {train_root.resolve()}'
assert test_root.exists(), f'test introuvable: {test_root.resolve()}'

def test_sort_key(path_obj: Path):
    stem = path_obj.stem
    return (0, int(stem)) if stem.isdigit() else (1, stem)

class TestImageDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir = Path(root_dir)
        self.transform = transform
        self.image_paths = sorted(list(self.root_dir.glob('*.jpg')), key=test_sort_key)
        if len(self.image_paths) == 0:
            self.image_paths = sorted(list(self.root_dir.rglob('*.jpg')), key=test_sort_key)
        self.imgnames = [p.name for p in self.image_paths]

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, index):
        path = self.image_paths[index]
        image = Image.open(path).convert('RGB')
        if self.transform is not None:
            image = self.transform(image)
        return image, self.imgnames[index]

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(CFG.img_size, scale=(0.60, 1.0)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.15),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.RandAugment(num_ops=2, magnitude=7),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    transforms.RandomErasing(p=0.15, scale=(0.02, 0.10), ratio=(0.3, 3.3)),
])

val_tf = transforms.Compose([
    transforms.Resize(int(CFG.img_size * 1.15)),
    transforms.CenterCrop(CFG.img_size),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

train_ds_full = datasets.ImageFolder(train_root, transform=train_tf)
val_ds_full = datasets.ImageFolder(train_root, transform=val_tf)
test_ds = TestImageDataset(test_root, transform=val_tf)

class_names = train_ds_full.classes
class_to_idx = train_ds_full.class_to_idx
idx_to_class = {v: k for k, v in class_to_idx.items()}
num_classes = len(class_names)

print(f'Train images: {len(train_ds_full):,}')
print(f'Test images : {len(test_ds):,}')
print(f'Classes     : {num_classes}')
print('Exemples classes:', class_names[:10])

Train images: 1,093,785
Test images : 121,428
Classes     : 86
Exemples classes: ['000_Physonectae', '001_tail_Appendicularia', '002_other_Siphonophorae', '003_Harpacticoida', '004_empty_Harpacticoida', '005_comb_Ctenophora', '006_larvae_Echinodermata', '007_Noctiluca', '008_detritus', '009_empty_Copepoda']


In [4]:
def mixup(images, targets, alpha=0.3):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    idx = torch.randperm(images.size(0), device=images.device)
    mixed = lam * images + (1 - lam) * images[idx]
    y_a, y_b = targets, targets[idx]
    return mixed, y_a, y_b, lam

def macro_f1(y_true, y_pred):
    return f1_score(y_true, y_pred, average='macro')

In [5]:
# Perte robuste au déséquilibre
class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, weight=self.alpha, reduction='none')
        pt = torch.exp(-ce)
        loss = ((1 - pt) ** self.gamma) * ce
        return loss.mean()

def make_model(num_classes):
    weights = models.EfficientNet_V2_S_Weights.DEFAULT
    model = models.efficientnet_v2_s(weights=weights)
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Linear(in_features, num_classes)
    return model

In [6]:
def build_loaders(train_indices, val_indices):
    train_subset = Subset(train_ds_full, train_indices)
    val_subset = Subset(val_ds_full, val_indices)

    train_targets = [train_ds_full.samples[i][1] for i in train_indices]
    class_counts = Counter(train_targets)
    class_weights = {cls: 1.0 / max(1, count) for cls, count in class_counts.items()}
    sample_weights = [class_weights[t] for t in train_targets]
    sampler = WeightedRandomSampler(
        weights=torch.as_tensor(sample_weights, dtype=torch.double),
        num_samples=len(sample_weights),
        replacement=True,
    )

    loader_kwargs = dict(
        num_workers=CFG.num_workers,
        pin_memory=(device.type == 'cuda'),
        persistent_workers=(CFG.num_workers > 0),
    )

    train_loader = DataLoader(
        train_subset,
        batch_size=CFG.batch_size,
        sampler=sampler,
        **loader_kwargs,
    )
    val_loader = DataLoader(
        val_subset,
        batch_size=CFG.batch_size,
        shuffle=False,
        **loader_kwargs,
    )
    return train_loader, val_loader, train_targets

def run_epoch_train(model, loader, optimizer, criterion, scaler):
    model.train()
    running_loss = 0.0
    optimizer.zero_grad(set_to_none=True)

    pbar = tqdm(loader, leave=False, desc='train')
    for step, (images, targets) in enumerate(pbar, start=1):
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        with torch.autocast(device_type='cuda', dtype=torch.float16):
            if CFG.use_mixup:
                mixed, y_a, y_b, lam = mixup(images, targets, CFG.mixup_alpha)
                logits = model(mixed)
                loss = lam * criterion(logits, y_a) + (1 - lam) * criterion(logits, y_b)
            else:
                logits = model(images)
                loss = criterion(logits, targets)

            loss = loss / CFG.grad_accum_steps

        scaler.scale(loss).backward()

        if step % CFG.grad_accum_steps == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)

        running_loss += loss.item() * images.size(0) * CFG.grad_accum_steps
        pbar.set_postfix(loss=f'{loss.item() * CFG.grad_accum_steps:.4f}')

    return running_loss / len(loader.dataset)

@torch.no_grad()
def run_epoch_val(model, loader, criterion):
    model.eval()
    running_loss = 0.0
    preds_all, y_all = [], []

    pbar = tqdm(loader, leave=False, desc='val')
    for images, targets in pbar:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        with torch.autocast(device_type='cuda', dtype=torch.float16):
            logits = model(images)
            loss = criterion(logits, targets)

        running_loss += loss.item() * images.size(0)
        preds = torch.argmax(logits, dim=1)
        preds_all.append(preds.cpu().numpy())
        y_all.append(targets.cpu().numpy())

    y_true = np.concatenate(y_all)
    y_pred = np.concatenate(preds_all)
    return running_loss / len(loader.dataset), macro_f1(y_true, y_pred)

In [7]:
# Entraînement (pipeline stable type Plankton.ipynb + améliorations v2)
class_to_indices = defaultdict(list)
for sample_idx, (_, class_idx) in enumerate(train_ds_full.samples):
    class_to_indices[class_idx].append(sample_idx)

train_indices = []
val_indices = []
for class_idx, idxs in class_to_indices.items():
    random.shuffle(idxs)
    n_val = max(1, int(len(idxs) * CFG.val_ratio))
    val_indices.extend(idxs[:n_val])
    train_indices.extend(idxs[n_val:])

random.shuffle(train_indices)
random.shuffle(val_indices)
print(f'Train size: {len(train_indices)} | Val size: {len(val_indices)}')

train_loader, val_loader, train_targets = build_loaders(train_indices, val_indices)

# Warmup: vérifie immédiatement que les workers livrent des batches
t0 = time.time()
_ = next(iter(train_loader))
print(f'Warmup train loader OK en {time.time() - t0:.1f}s')
t0 = time.time()
_ = next(iter(val_loader))
print(f'Warmup val loader OK en {time.time() - t0:.1f}s')

model = make_model(num_classes).to(device)

counts = Counter(train_targets)
alpha = torch.ones(num_classes, dtype=torch.float32)
for c in range(num_classes):
    alpha[c] = 1.0 / max(1, counts.get(c, 1))
alpha = (alpha / alpha.mean()).to(device)

criterion = FocalLoss(alpha=alpha, gamma=2.0)
optimizer = torch.optim.AdamW(model.parameters(), lr=CFG.lr, weight_decay=CFG.weight_decay)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=max(1, CFG.epochs))
scaler = torch.amp.GradScaler('cuda')

best_f1 = -1.0
best_state = None
best_epoch = -1
no_improve = 0

for epoch in range(1, CFG.epochs + 1):
    tr_loss = run_epoch_train(model, train_loader, optimizer, criterion, scaler)
    va_loss, va_f1 = run_epoch_val(model, val_loader, criterion)
    scheduler.step()

    print(f'Epoch {epoch:02d}/{CFG.epochs} | tr_loss={tr_loss:.4f} | va_loss={va_loss:.4f} | va_f1={va_f1:.4f}')

    epoch_ckpt = Path(CFG.ckpt_dir) / f'epoch_{epoch:02d}_f1_{va_f1:.4f}.pt'
    torch.save({
        'state_dict': model.state_dict(),
        'class_to_idx': class_to_idx,
        'img_size': CFG.img_size,
        'epoch': epoch,
        'val_macro_f1': va_f1,
    }, epoch_ckpt)

    if va_f1 > best_f1:
        best_f1 = va_f1
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        best_epoch = epoch
        no_improve = 0
    else:
        no_improve += 1

    if no_improve >= CFG.early_stopping_patience:
        print('Early stopping')
        break

model.load_state_dict(best_state)

single_ckpt = Path(CFG.ckpt_dir) / f'single_best_f1_{best_f1:.4f}.pt'
torch.save({
    'state_dict': model.state_dict(),
    'class_to_idx': class_to_idx,
    'img_size': CFG.img_size,
    'best_f1': best_f1,
    'best_epoch': best_epoch
}, single_ckpt)

print('Best val macro-F1:', round(best_f1, 5))
print('Checkpoint saved:', single_ckpt)

Train size: 984444 | Val size: 109341
Warmup train loader OK en 15.2s
Warmup val loader OK en 12.9s


Epoch 01/14 | tr_loss=0.5741 | va_loss=0.0142 | va_f1=0.1090


KeyboardInterrupt: 

In [ ]:
# Inference test + TTA + création submission.csv
def tta_logits(model, images):
    logits = model(images)
    if CFG.tta:
        logits_h = model(torch.flip(images, dims=[3]))
        logits_v = model(torch.flip(images, dims=[2]))
        logits = (logits + logits_h + logits_v) / 3.0
    return logits

test_loader = DataLoader(
    test_ds,
    batch_size=256,
    shuffle=False,
    num_workers=0,
    pin_memory=True,
    persistent_workers=False,
    )

single_ckpts = sorted([str(p) for p in Path(CFG.ckpt_dir).glob('single_best_f1_*.pt')])
assert len(single_ckpts) > 0, 'Aucun checkpoint mono-modèle trouvé. Lance la cellule entraînement.'
ckpt = single_ckpts[-1]
print('loading', ckpt)

model = make_model(num_classes).to(device)
state = torch.load(ckpt, map_location='cpu')
model.load_state_dict(state['state_dict'])
model.eval()

rows = []
with torch.inference_mode():
    for images, imgnames in tqdm(test_loader, leave=False, desc='infer single model'):
        images = images.to(device, non_blocking=True)
        logits = tta_logits(model, images)
        preds = torch.argmax(torch.softmax(logits, dim=1), dim=1).cpu().numpy()
        rows.extend(zip(imgnames, preds.astype(int)))

rows = sorted(rows, key=lambda x: test_sort_key(Path(x[0])))
submission = pd.DataFrame(rows, columns=['imgname', 'label'])
submission.to_csv('submission.csv', index=False)
print('submission.csv généré')
submission.head()